# Ordinary Least Squares Regression Coefficients (Linear Algebra)

This notebook derives Ordinary Least Squares (OLS) regression coefficients 
manually using the matrix left inverse formula.
Most practical applications of OLS is through black-box libraries. This notebook strips that away and derives the coefficients directly from first principles, specifically the closed-form linar algebra solution to OLS coefficients:

**β = (XᵀX)⁻¹Xᵀy**

**X** is the design matrix (matrix of data points with each column representing the independent variables) and **y** is the target vector (dependent variable). The last part also includes the same coefficients calculated using the standard statistical library `sklearn`.
Note: the derivation of this left inverse mathematical formula is outside the scope of this notebook.

## The data
The design matrix is constructed from 5 years of daily price return data, 
structured as an autoregressive model with 5 lags (AR(5)). Each row 
represents a point in time, and each column represents a lagged return, 
meaning we are using the past 5 returns data points to predict the next return. 
Note: Time-series analysis is not covered in detail in this notebook.

This is the same mathematical structure underlying:

- Factor investing in quantitative finance
- The linear layer in neural networks

In [7]:
# import libraries
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [8]:
# download historical data for SP500 from Yahoo Finance
ticker = '^GSPC'
period = '5y'
data = yf.download(ticker, period = period, multi_level_index = False)

# drop unnecessary columns and calculate returns
data['Returns'] = data['Close'].pct_change()
data = data[['Close', 'Returns']].dropna()

# create autoregressive features (lag = 5 design matrix)
lags = 5
X = pd.DataFrame(index = data.index)

X['intercept'] = 1 # add intercept term 
for lag in range(1, lags + 1):
    X[f'Lag_{lag}'] = data['Returns'].shift(lag)

X = X.dropna()

# target variable
y = pd.DataFrame(data['Returns'].loc[X.index])
y.columns = ['target']

# combine features and target into a single table for inspection
table = pd.concat([X, y], axis=1)
table.head()

[*********************100%***********************]  1 of 1 completed


,intercept,Lag_1,Lag_2,Lag_3,Lag_4,Lag_5,target
Date,,,,,,,
2021-03-23,1,0.007025,-0.000603,-0.014761,0.002879,-0.001570,-0.007631
2021-03-24,1,-0.007631,0.007025,-0.000603,-0.014761,0.002879,-0.005467
2021-03-25,1,-0.005467,-0.007631,0.007025,-0.000603,-0.014761,0.005240
2021-03-26,1,0.005240,-0.005467,-0.007631,0.007025,-0.000603,0.016631
2021-03-29,1,0.016631,0.005240,-0.005467,-0.007631,0.007025,-0.000868


In [18]:
# compute OLS estimates using matrix algebra
beta = np.linalg.solve(X.T @ X, X.T @ y)
# flatten to 1D array so we can compare element-wise with sklearn's coef_
beta = np.array(beta).ravel()

# coefficients using sklearn for comparison
model = LinearRegression(fit_intercept=False)
model.fit(X, y)
sklearn_coef = model.coef_.ravel()

# print coefficients and check for errors
print(f'OLS inverse intercept = {np.round(beta[0], 4)}')

for i in range(1, len(beta)):
    print(f'OLS inverse beta_{i} = {np.round(beta[i], 4)}')

print('-------')
print(f'Sklearn intercept = {np.round(sklearn_coef[0], 4)}')

for i in range(1, len(sklearn_coef)):
    print(f'Sklearn beta_{i} = {np.round(sklearn_coef[i], 4)}')

print('-------')
print("Difference between OLS and sklearn coefficients:")
print(np.round(sklearn_coef - beta, 2))


OLS inverse intercept = 0.0005
OLS inverse beta_1 = -0.024
OLS inverse beta_2 = -0.006
OLS inverse beta_3 = -0.0697
OLS inverse beta_4 = -0.042
OLS inverse beta_5 = -0.0043
-------
Sklearn intercept = 0.0005
Sklearn beta_1 = -0.024
Sklearn beta_2 = -0.006
Sklearn beta_3 = -0.0697
Sklearn beta_4 = -0.042
Sklearn beta_5 = -0.0043
-------
Difference between OLS and sklearn coefficients:
[ 0.  0. -0. -0. -0.  0.]
